In [ ]:
!pip install -U \
  torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  transformers==4.51.0 \
  bitsandbytes \
  peft==0.15.2 \
  trl==0.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import re
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer #, SFTTrainingArguments
from trl import setup_chat_format
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, auc
)

In [ ]:
print(f"pytorch version {torch.__version__}")

pytorch version 2.6.0+cu124


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

working on cuda:0


In [ ]:
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

### Importing the Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainset = pd.read_csv('/content/drive/My Drive/Thesis/Trainset/train_data_df.csv')
testset = pd.read_csv('/content/drive/My Drive/Thesis/Testset/test_data_df.csv')

In [ ]:
train = trainset[['news_content', 'impact_length']]
test = testset[['news_content', 'impact_length']]

In [ ]:
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=trainset["impact_length"])

### Functions


In [ ]:
def generate_prompt(data_point):
    return f"""
            Analyze the impact length of the news content.
            The impact length reflects how long-lasting the ESG-related consequences of the news are expected to be.
            Classify the news as having a Less than 2 years, 2 to 5 years, or More than 5 years impact based on its content.
            Return only the corresponding numerical label: 0 for Less than 2 years impact, 1 for 2 to 5 years impact, 2 for More than 5 years impact

            [{data_point["news_content"]}] = {data_point["impact_length"]}
            """.strip()

def generate_test_prompt(data_point):
    return f"""
            Analyze the impact length of the news content.
            The impact length reflects how long-lasting the ESG-related consequences of the news are expected to be.
            Classify the news as having a Less than 2 years, 2 to 5 years, or More than 5 years impact based on its content.
            Return only the corresponding numerical label: 0 for Less than 2 years impact, 1 for 2 to 5 years impact, 2 for More than 5 years impact

            [{data_point["news_content"]}] = """.strip()

def evaluate(y_true, y_pred, y_pred_proba=None):
    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Overall Accuracy: {accuracy:.3f}')

    unique_labels = sorted(set(y_true))
    for label in unique_labels:
        label_indices = [i for i in range(len(y_true)) if y_true[i] == label]
        label_y_true = [y_true[i] for i in label_indices]
        label_y_pred = [y_pred[i] for i in label_indices]
        label_accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {label}: {label_accuracy:.3f}')

    f1_macro = f1_score(y_true, y_pred, average='macro')
    print(f'Macro F1-score: {f1_macro:.3f}')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, digits=3))

    print('\nConfusion Matrix:')
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

    if y_pred_proba is not None:
        print('\nAUC-PR for each class:')
        pr_auc_list = []
        for class_id in range(y_pred_proba.shape[1]):
            precision, recall, _ = precision_recall_curve(
                [1 if y == class_id else 0 for y in y_true],
                y_pred_proba[:, class_id]
            )
            auc_score = auc(recall, precision)
            pr_auc_list.append(auc_score)
            print(f'Class {class_id}: AUC-PR = {auc_score:.3f}')

        macro_auc_pr = np.mean(pr_auc_list)
        print(f'Macro AUC-PR: {macro_auc_pr:.3f}')

In [ ]:
from huggingface_hub import login
login(token="...")

In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

max_seq_length = 512
tokenizer = AutoTokenizer.from_pretrained(model_name, max_seq_length=max_seq_length)
tokenizer.pad_token_id = tokenizer.eos_token_id

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
def predict(test, model, tokenizer):
    y_pred = []

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["news_content"]
        prompt = (
            "Analyze the impact length of the news content inside the [].\n"
            "Return ONLY the numerical label (0 for Less than 2 years, 1 for 2 to 5 years, 2 for More than 5 years).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[A large oil spill occurred in the Gulf of Mexico.] = 2\n\n"
            f"[{content}] ="
        )

        pipe = pipeline(
            task="text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=10,
            do_sample=False  # Greedy decoding
        )

        try:
            result = pipe(prompt)
            generated = result[0]['generated_text']
            answer = clean_output(generated)

            if answer is None:
                print(f"[{i}] Unexpected answer: '{generated}'")
                answer = 1

            y_pred.append(answer)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback

    return y_pred

In [ ]:
def clean_output(output):
    match = re.search(r'\b([0-2])\b', output)
    if match:
        return int(match.group(1))
    return None

def predict(test, model, tokenizer):
    y_pred = []

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=10,
        do_sample=False  # Greedy decoding
    )

    for i in tqdm(range(len(test))):
        content = test.iloc[i]["news_content"]
        prompt = (
            "Analyze the impact length of the news content inside the [].\n"
            "Classify the impact length of the following news content as 0 (Less than 2 years), 1 (2 to 5 years), or 2 (More than 5 years):\n\n"
            "Return ONLY the numerical label (0 for Less than 2 years, 1 for 2 to 5 years, 2 for More than 5 years).\n"
            "No explanation, no additional text.\n"
            "Example:\n"
            "[A large oil spill occurred in the Gulf of Mexico.] = 2\n"
            f"[{content}]\n\nAnswer:"
        )

        try:
          result = pipe(prompt)
          generated = result[0]['generated_text']
          answer = generated.split("Answer:")[-1].strip()

          cleaned = clean_output(answer)
          print(f"[{i}] Raw: '{answer}' → Cleaned: {cleaned}")

          if cleaned is None:
            cleaned = 1  # Fallback if regex fails

          y_pred.append(cleaned)

        except Exception as e:
            print(f"Error at index {i}: {e}")
            y_pred.append(1)  # fallback in case of generation error

    return y_pred

## Apply the functions

In [ ]:
X_train = pd.DataFrame(train_df.apply(generate_prompt, axis=1),
                       columns=["news_content"])
X_val = pd.DataFrame(val_df.apply(generate_prompt, axis=1),
                      columns=["news_content"])
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1),
                      columns=["news_content"])

y_true = test.impact_length

train_data = Dataset.from_pandas(X_train)
val_data = Dataset.from_pandas(X_val)

In [ ]:
sample = train_data[0]  # Get the first item
print(sample)

{'news_content': 'Analyze the impact length of the news content.\n            The impact length reflects how long-lasting the ESG-related consequences of the news are expected to be.\n            Classify the news as having a Less than 2 years, 2 to 5 years, or More than 5 years impact based on its content.\n            Return only the corresponding numerical label: 0 for Less than 2 years impact, 1 for 2 to 5 years impact, 2 for More than 5 years impact\n\n            [This event has also hit us at a point where we’ve seen unresolved challenges from sustained high prices for agricultural commodities and fertilizers since late 2020. We’ve seen corn prices, for example, at well above $5 per bushel since early 2021—well over a year now.] = 1', '__index_level_0__': 313}


In [ ]:
y_pred = predict(test, model, tokenizer)

Device set to use cuda:0
  1%|          | 1/136 [00:02<04:55,  2.19s/it]

[0] Raw: '1
[The first COVID-19 vaccine' → Cleaned: 1


  1%|▏         | 2/136 [00:03<03:09,  1.42s/it]

[1] Raw: '1
[The COVID-19 pandemic has' → Cleaned: 1


  2%|▏         | 3/136 [00:03<02:34,  1.16s/it]

...

[133] Raw: '2
[The COVID-19 pandemic,' → Cleaned: 2


 99%|█████████▉| 135/136 [02:08<00:01,  1.02s/it]

[134] Raw: '1
[The COVID-19 pandemic has' → Cleaned: 1


100%|██████████| 136/136 [02:09<00:00,  1.05it/s]

[135] Raw: '1
[The COVID-19 pandemic has' → Cleaned: 1


In [ ]:
evaluate(y_true, y_pred)

Overall Accuracy: 0.324
Accuracy for label 0: 0.000
Accuracy for label 1: 0.362
Accuracy for label 2: 0.325
Macro F1-score: 0.254

Classification Report:
              precision    recall  f1-score   support

           0      0.000     0.000     0.000         6
           1      0.298     0.362     0.327        47
           2      0.659     0.325     0.435        83

    accuracy                          0.324       136
   macro avg      0.319     0.229     0.254       136
weighted avg      0.505     0.324     0.379       136


Confusion Matrix:
[[ 0  4  2]
 [18 17 12]
 [20 36 27]]


### Fine Tuning

In [ ]:
login(token="...")

In [ ]:
output_dir="trained_weigths"
modules = ["q_proj", "v_proj"]
output_dir="/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model"
hub_model_id = "Adrakou/llama-3.1-fine-tuned"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=False,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps = 50,
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_strategy="end",
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    dataset_text_field="news_content",
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    packing=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    }
)

Map:   0%|          | 0/436 [00:00<?, ? examples/s]

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
50,1.043500,1.131618
100,1.030900,1.059247
150,1.117600,1.059100
200,0.851700,1.081313
250,0.838100,1.107648
300,0.692300,1.144081
350,0.686200,1.163231
400,0.525200,1.204983
450,0.685400,1.220977
500,0.559000,1.222567


TrainOutput(global_step=540, training_loss=0.8711321850065832, metrics={'train_runtime': 4688.4958, 'train_samples_per_second': 0.93, 'train_steps_per_second': 0.115, 'total_flos': 3.5764110040203264e+16, 'train_loss': 0.8711321850065832, 'epoch': 9.825688073394495})

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(output_dir)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uned-model/tokenizer.json:  97%|#########7| 16.7MB / 17.2MB            

  ...adapter_model.safetensors:   4%|4         |  555kB / 13.6MB            

  ...d-model/training_args.bin:   4%|4         |   218B / 5.37kB            

('/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer_config.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/special_tokens_map.json',
 '/content/drive/My Drive/Thesis/llama-3.1-fine-tuned-model/tokenizer.json')

In [ ]:
trainer.push_to_hub()
tokenizer.push_to_hub(hub_model_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d-model/training_args.bin: 100%|##########| 5.37kB / 5.37kB            

  ...adapter_model.safetensors: 100%|##########| 13.6MB / 13.6MB            

  ...uned-model/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpb27tdf7m/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Adrakou/llama-3.1-fine-tuned/commit/c144bb9cacefca28a51ab20515af00794548b337', commit_message='Upload tokenizer', commit_description='', oid='c144bb9cacefca28a51ab20515af00794548b337', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Adrakou/llama-3.1-fine-tuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Adrakou/llama-3.1-fine-tuned'), pr_revision=None, pr_num=None)

In [ ]:
y_pred = predict(test, model, tokenizer)
evaluate(y_true, y_pred)

Device set to use cuda:0
  1%|          | 1/136 [00:01<04:19,  1.92s/it]

[0] Raw: '2
[The new index will be calculated' → Cleaned: 2


  1%|▏         | 2/136 [00:03<03:35,  1.61s/it]

[1] Raw: '1
[The new partnership will see the' → Cleaned: 1


  2%|▏         | 3/136 [00:04<03:18,  1.49s/it]

...

[133] Raw: '2
[The new agreement will see A' → Cleaned: 2


 99%|█████████▉| 135/136 [03:20<00:01,  1.59s/it]

[134] Raw: '2
[The new regulations will require companies' → Cleaned: 2


100%|██████████| 136/136 [03:21<00:00,  1.48s/it]

[135] Raw: '2
[The company also announced the appointment' → Cleaned: 2
Overall Accuracy: 0.522
Accuracy for label 0: 0.000
Accuracy for label 1: 0.489
Accuracy for label 2: 0.578
Macro F1-score: 0.355

Classification Report:
              precision    recall  f1-score   support

           0      0.000     0.000     0.000         6
           1      0.418     0.489     0.451        47
           2      0.658     0.578     0.615        83

    accuracy                          0.522       136
   macro avg      0.359     0.356     0.355       136
weighted avg      0.546     0.522     0.531       136


Confusion Matrix:
[[ 0  3  3]
 [ 2 23 22]
 [ 6 29 48]]


In [ ]:
#old epoch = 1 f1=0.218
#old epoch = 3
#y_pred = predict(test, model, tokenizer)
#evaluate(y_true, y_pred)

In [ ]:
evaluation = pd.DataFrame({'text': X_test["news_content"],
                           'y_true':y_true,
                           'y_pred': y_pred},
                         )
evaluation.to_csv("test_predictions.csv", index=False)

In [ ]:
print(evaluation)

                                                  text  y_true  y_pred
0    Analyze the impact length of the news content....       1       2
1    Analyze the impact length of the news content....       1       2
2    Analyze the impact length of the news content....       1       2
3    Analyze the impact length of the news content....       1       2
4    Analyze the impact length of the news content....       1       1
..                                                 ...     ...     ...
131  Analyze the impact length of the news content....       2       2
132  Analyze the impact length of the news content....       2       2
133  Analyze the impact length of the news content....       2       2
134  Analyze the impact length of the news content....       2       2
135  Analyze the impact length of the news content....       2       2

[136 rows x 3 columns]
